# Calibration Targets and Fitted Parameters


This notebook introduces the calibration-target side of `cqed_sim`. We use lightweight spectroscopy, Rabi, and T1 target helpers to generate effective calibration traces, then read back the fitted device parameters.

The physical role of this notebook is not pulse-level realism. It is a workflow bridge: these targets synthesize the summary data products that a later system-identification or control stack would consume, so you can reason clearly about what is being estimated and in what units.


## Imports


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from tutorials.workflow_tutorial_support import configure_notebook_style

configure_notebook_style()

from cqed_sim import DispersiveTransmonCavityModel, run_rabi, run_spectroscopy, run_t1
from tutorials.tutorial_support import GHz, MHz, ns, us


## Physics / model definition


In [ ]:
model = DispersiveTransmonCavityModel(
    omega_c=GHz(5.0),
    omega_q=GHz(6.0),
    alpha=MHz(-220.0),
    chi=MHz(-2.25),
    kerr=MHz(-0.006),
    n_cav=8,
    n_tr=2,
)

drive_frequencies_hz = np.linspace(5.9e9, 6.1e9, 121)
drive_frequencies = 2.0 * np.pi * drive_frequencies_hz
rabi_amplitudes = np.linspace(0.0, 2.0e8, 61)
t1_delays = np.linspace(0.0, 30.0, 61) * us


## Pulse / sequence construction


In [ ]:
print("These helpers synthesize effective calibration traces and fitted summaries; they are intended for calibration / system-ID workflows rather than full pulse-level experiment replay.")


## Simulation


In [ ]:
spectroscopy = run_spectroscopy(model, drive_frequencies)
rabi = run_rabi(model, rabi_amplitudes, duration=40.0 * ns)
t1 = run_t1(model, t1_delays, t1=24.0 * us)

omega_01_fit_ghz = spectroscopy.fitted_parameters["omega_01"] / (2.0 * np.pi * 1.0e9)
omega_12_fit_ghz = spectroscopy.fitted_parameters["omega_12"] / (2.0 * np.pi * 1.0e9)
omega_scale_fit = float(rabi.fitted_parameters["omega_scale"])
t1_fit_us = t1.fitted_parameters["t1"] / us

print("omega_01 fit [GHz]:", omega_01_fit_ghz)
print("omega_12 fit [GHz]:", omega_12_fit_ghz)
print("Rabi omega_scale fit:", omega_scale_fit)
print("T1 fit [us]:", t1_fit_us)


## Analysis / visualization


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.0, 3.8))

axes[0].plot(drive_frequencies_hz / 1.0e9, spectroscopy.raw_data["response"], color="#4C78A8")
axes[0].axvline(omega_01_fit_ghz, color="#E45756", ls="--", lw=1.0, label=r"$\omega_{01}$ fit")
axes[0].axvline(omega_12_fit_ghz, color="#54A24B", ls=":", lw=1.0, label=r"$\omega_{12}$ fit")
axes[0].set_xlabel("Drive frequency (GHz)")
axes[0].set_ylabel("Effective spectroscopy response")
axes[0].set_title("Spectroscopy target")
axes[0].legend(loc="upper right")

axes[1].plot(rabi.raw_data["amplitudes"] / 1.0e8, rabi.raw_data["excited_population"], color="#F58518")
axes[1].set_xlabel(r"Drive amplitude ($10^8$ rad/s)")
axes[1].set_ylabel(r"Excited population $P_e$")
axes[1].set_title(rf"Rabi target, fitted scale = {omega_scale_fit:.3f}")

axes[2].plot(t1.raw_data["delays"] / us, t1.raw_data["excited_population"], color="#72B7B2")
axes[2].set_xlabel("Delay (us)")
axes[2].set_ylabel(r"Excited population $P_e$")
axes[2].set_title(rf"T1 target, fitted $T_1 = {t1_fit_us:.2f}$ us")

plt.tight_layout()
plt.show()


## Interpretation


Each helper returns a `CalibrationResult` with fitted parameters, uncertainties, raw synthetic data, and metadata. The point is not that these traces are the only valid calibration procedure, but that they create a consistent data structure for downstream fitting and uncertainty propagation.

For system-identification work, the fitted summaries are often more important than the raw traces. They become the posteriors or priors that later randomization and robust-control workflows will consume.


## Variations / exercises


- Change the spectroscopy span or linewidth and see how the fitted uncertainty changes.
- Repeat the Rabi target with a different duration to understand the fitted `omega_scale` normalization.
- Feed these fitted results into the next notebook to build calibration-informed randomization priors.
